In [237]:
import pandas as pd
import seaborn as sns   
import matplotlib.pyplot as plt
import plotly.express as px
import hashlib
import numpy as np
import missingno as msno
import os
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder

Importamos los excels y creamos el dataframe de cada uno

In [238]:
dic=pd.read_excel("data/Diccionario_Variables_Alumnos.xlsx")
dic

,Variables,Descripción,Valores
0,ID,Identificador único del préstamo.,Código identificador del cliente.
1,Edad,Edad del prestatario.,Valor numérico.
2,Ingresos,Ingreso anual del prestatario.,Valor numérico.
3,Monto_Inicial,Cantidad de dinero acordada.,Valor numérico.
4,Scoring_Crediticio,"La puntuación crediticia del prestatario, que ...",Valor numérico.
5,Meses_Empleo,Meses de vida laboral del prestatario.,Valor numérico.
6,Num_Creditos,Número de líneas de credito abiertas del prest...,Valor numérico.
7,Ratio_Interes,El ratio de interés del préstamo (multiplicado...,Valor numérico.
8,Duracion,La duración del prestamo en meses.,Valor numérico.
9,Ratio_Deuda_Ingresos,"Ratio de deuda frente a ingresos, que indica l...",Valor numérico.


In [239]:
df=pd.read_excel("data/Prestamos_Data_Alumnos_v5.xlsx")
df

,ID,Edad,Ingresos,Monto_Inicial,Scoring_Crediticio,Meses_Empleo,Num_Creditos,Ratio_Interes,Duracion,Ratio_Deuda_Ingresos,Estudios,Tipo_Jornada_Laboral,Estado_Civil,Posesion_Hipoteca,Personas_Cargo,Proposito,Fiador,Impago,Prima
0,O77MPJ,26,19153.0,6973,706,9.0,1,19.05,36.0,0.10,Grado Universitario,Jornada completa,Soltero,0.0,1.0,Automóvil,1.0,1,130.83
1,6B4H6E,22,16000.0,10000,769,22.0,1,8.96,48.0,0.32,Escolar,Desempleado,Soltero,1.0,1.0,Vivienda,0.0,0,65.85
2,VKTUXP,33,38308.0,16474,699,70.0,3,14.07,36.0,0.20,Máster,Jornada completa,Casado,1.0,1.0,Vivienda,0.0,0,260.60
3,5P9OVL,51,51271.0,40770,812,309.0,1,12.66,12.0,0.36,Grado Universitario,Jornada completa,Casado,0.0,1.0,Vivienda,1.0,0,221.02
4,KG9CT9,34,56572.0,73978,763,86.0,1,13.91,60.0,0.55,Máster,Tiempo parcial,Soltero,0.0,0.0,Educación,0.0,0,800.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
510699,MY63P5,28,17390.0,5000,744,92.0,3,16.12,12.0,0.10,Escolar,Desempleado,Soltero,0.0,0.0,Negocios,0.0,0,26.80
510700,6903JQ,42,33486.0,33048,760,206.0,1,9.09,24.0,0.31,Grado Universitario,Autónomo,Divorciado,0.0,0.0,Automóvil,0.0,0,172.41
510701,EPB8QX,64,40369.0,40000,680,447.0,2,7.75,12.0,0.19,Escolar,Jornada completa,Casado,1.0,1.0,Negocios,1.0,1,207.42
510702,R1QRKW,35,30915.0,130966,739,134.0,2,10.22,24.0,0.55,Escolar,Jornada completa,Casado,1.0,0.0,Educación,0.0,0,580.00


Filtramos el dataframe para que contenga unicamente los datos de nuestro grupo (negocios)

In [240]:
df1 = df[df["Proposito"] == "Negocios"].reset_index(drop=True)
df1 = df1.drop(columns=["Proposito"])

Tratamiento de datos

In [241]:
filtrado = df1.copy()

In [242]:
filtrado.info()

<class 'pandas.DataFrame'>
RangeIndex: 76515 entries, 0 to 76514
Data columns (total 18 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   ID                    76515 non-null  str    
 1   Edad                  76515 non-null  int64  
 2   Ingresos              76515 non-null  float64
 3   Monto_Inicial         76515 non-null  int64  
 4   Scoring_Crediticio    76515 non-null  int64  
 5   Meses_Empleo          76515 non-null  float64
 6   Num_Creditos          76515 non-null  int64  
 7   Ratio_Interes         76515 non-null  float64
 8   Duracion              76514 non-null  float64
 9   Ratio_Deuda_Ingresos  76515 non-null  float64
 10  Estudios              76515 non-null  str    
 11  Tipo_Jornada_Laboral  76515 non-null  str    
 12  Estado_Civil          76515 non-null  str    
 13  Posesion_Hipoteca     76515 non-null  float64
 14  Personas_Cargo        76515 non-null  float64
 15  Fiador                76514 no

In [243]:
filtrado.describe()

,Edad,Ingresos,Monto_Inicial,Scoring_Crediticio,Meses_Empleo,Num_Creditos,Ratio_Interes,Duracion,Ratio_Deuda_Ingresos,Posesion_Hipoteca,Personas_Cargo,Fiador,Impago,Prima
count,76515.000000,76515.000000,76515.000000,76515.000000,76515.000000,76515.000000,76515.000000,76514.000000,76515.000000,76515.000000,76515.000000,76514.000000,76515.000000,76514.000000
mean,39.648958,41470.754192,45120.381063,692.537280,212.438319,1.999347,10.661792,37.164963,0.321617,0.349853,0.449454,0.299749,0.115794,384.775398
std,10.077878,15917.494176,46683.672913,87.672823,150.752051,0.998862,3.800952,14.182988,0.180931,0.476926,0.497442,0.458151,0.319980,267.323226
min,19.000000,16000.000000,3000.000000,398.000000,0.000000,1.000000,2.010000,12.000000,0.100000,0.000000,0.000000,0.000000,0.000000,10.000000
25%,33.000000,30485.000000,15023.000000,636.000000,69.000000,1.000000,7.980000,24.000000,0.140000,0.000000,0.000000,0.000000,0.000000,137.270000
50%,39.000000,39093.000000,40000.000000,704.000000,207.000000,2.000000,10.590000,36.000000,0.290000,0.000000,0.000000,0.000000,0.000000,335.945000
75%,46.000000,50043.000000,47716.000000,761.000000,337.000000,3.000000,13.220000,48.000000,0.550000,1.000000,1.000000,1.000000,0.000000,612.872500
max,64.000000,120000.000000,400000.000000,845.000000,639.000000,4.000000,24.440000,60.000000,0.550000,1.000000,1.000000,1.000000,1.000000,800.000000


In [244]:
filtrado.shape

(76515, 18)

In [245]:
filtrado.columns.unique()

Index(['ID', 'Edad', 'Ingresos', 'Monto_Inicial', 'Scoring_Crediticio',
       'Meses_Empleo', 'Num_Creditos', 'Ratio_Interes', 'Duracion',
       'Ratio_Deuda_Ingresos', 'Estudios', 'Tipo_Jornada_Laboral',
       'Estado_Civil', 'Posesion_Hipoteca', 'Personas_Cargo', 'Fiador',
       'Impago', 'Prima'],
      dtype='str')

In [246]:
filtrado.isna().sum().sum()

np.int64(3)

In [247]:
null = filtrado[filtrado.isna().any(axis=1)]
null

,ID,Edad,Ingresos,Monto_Inicial,Scoring_Crediticio,Meses_Empleo,Num_Creditos,Ratio_Interes,Duracion,Ratio_Deuda_Ingresos,Estudios,Tipo_Jornada_Laboral,Estado_Civil,Posesion_Hipoteca,Personas_Cargo,Fiador,Impago,Prima
21337,UKOIYP,36,55314.0,12966,660,111.0,2,8.25,NaN,0.10,Doctorado,Tiempo parcial,Casado,1.0,1.0,1.0,0,NaN
53732,Q4YH86,43,44639.0,40795,447,311.0,2,7.60,12.0,0.22,Escolar,Tiempo parcial,Casado,1.0,1.0,NaN,0,245.79


In [248]:
filtrado = filtrado.dropna(axis=0, how='any')

In [249]:
filtrado.duplicated().sum()

np.int64(3)

In [250]:
filtrado.duplicated().sum()

np.int64(3)

In [251]:
filtrado = filtrado.drop_duplicates()

In [252]:
filtrado

,ID,Edad,Ingresos,Monto_Inicial,Scoring_Crediticio,Meses_Empleo,Num_Creditos,Ratio_Interes,Duracion,Ratio_Deuda_Ingresos,Estudios,Tipo_Jornada_Laboral,Estado_Civil,Posesion_Hipoteca,Personas_Cargo,Fiador,Impago,Prima
0,JKLXIQ,51,56857.0,102886,718,338.0,3,6.18,24.0,0.55,Grado Universitario,Desempleado,Casado,1.0,0.0,0.0,0,580.00
1,4UO7DJ,46,45840.0,30473,689,291.0,1,14.28,36.0,0.34,Escolar,Desempleado,Casado,0.0,0.0,0.0,0,431.25
2,MO3OJW,36,48156.0,188824,675,161.0,2,13.35,36.0,0.55,Grado Universitario,Jornada completa,Soltero,0.0,1.0,0.0,0,760.00
3,I14TTP,42,41472.0,40000,582,312.0,1,11.83,12.0,0.44,Escolar,Jornada completa,Divorciado,1.0,1.0,1.0,0,385.66
4,8FUIBH,60,70202.0,40743,652,489.0,2,9.27,48.0,0.25,Grado Universitario,Tiempo parcial,Casado,1.0,0.0,1.0,0,535.14
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
76510,SXTHNM,24,17942.0,40000,823,35.0,2,12.52,36.0,0.55,Escolar,Desempleado,Soltero,0.0,1.0,0.0,1,540.62
76511,0U329X,54,35399.0,14989,709,445.0,1,5.93,48.0,0.47,Escolar,Tiempo parcial,Soltero,0.0,1.0,0.0,1,164.30
76512,MY63P5,28,17390.0,5000,744,92.0,3,16.12,12.0,0.10,Escolar,Desempleado,Soltero,0.0,0.0,0.0,0,26.80
76513,EPB8QX,64,40369.0,40000,680,447.0,2,7.75,12.0,0.19,Escolar,Jornada completa,Casado,1.0,1.0,1.0,1,207.42


In [253]:
df.shape[0]

510704

In [254]:
filtrado.shape[0]

76510

In [255]:
total = filtrado.shape[0]/df.shape[0]
total


0.14981280741877878

#Cambio de variables categoricas a binarias para su futuro uso en modelos mediante one hot encoding y ordinal encoding

In [256]:
# 1. Definir el encoder indicando que queremos un array denso directamente
onehot_enc = OneHotEncoder(sparse_output=False)

# 2. Ajustar y transformar las variables nominales
columnas_categoricas = ["Estado_Civil", "Tipo_Jornada_Laboral"]
datos_onehot = onehot_enc.fit_transform(filtrado[columnas_categoricas])

# 3. Obtener los nombres de las nuevas columnas
feature_names = onehot_enc.get_feature_names_out(input_features=columnas_categoricas)

# 4. Creacion de un DataFrame con los datos codificados y usamos la columna "ID" para unirlo con el DataFrame original
data_encoded = pd.DataFrame(datos_onehot, columns=feature_names)
data_encoded["ID"] = filtrado["ID"].values

# 5. Unimos ambos DataFrames usando pd.merge() en base a la columna "ID"
datos = pd.merge(filtrado.drop(columns=columnas_categoricas), data_encoded, on="ID")


In [257]:
data_encoded

,Estado_Civil_Casado,Estado_Civil_Divorciado,Estado_Civil_Soltero,Tipo_Jornada_Laboral_Autónomo,Tipo_Jornada_Laboral_Desempleado,Tipo_Jornada_Laboral_Jornada completa,Tipo_Jornada_Laboral_Tiempo parcial,ID
0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,JKLXIQ
1,1.0,0.0,0.0,0.0,1.0,0.0,0.0,4UO7DJ
2,0.0,0.0,1.0,0.0,0.0,1.0,0.0,MO3OJW
3,0.0,1.0,0.0,0.0,0.0,1.0,0.0,I14TTP
4,1.0,0.0,0.0,0.0,0.0,0.0,1.0,8FUIBH
...,...,...,...,...,...,...,...,...
76505,0.0,0.0,1.0,0.0,1.0,0.0,0.0,SXTHNM
76506,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0U329X
76507,0.0,0.0,1.0,0.0,1.0,0.0,0.0,MY63P5
76508,1.0,0.0,0.0,0.0,0.0,1.0,0.0,EPB8QX


In [258]:
datos

,ID,Edad,Ingresos,Monto_Inicial,Scoring_Crediticio,Meses_Empleo,Num_Creditos,Ratio_Interes,Duracion,Ratio_Deuda_Ingresos,...,Fiador,Impago,Prima,Estado_Civil_Casado,Estado_Civil_Divorciado,Estado_Civil_Soltero,Tipo_Jornada_Laboral_Autónomo,Tipo_Jornada_Laboral_Desempleado,Tipo_Jornada_Laboral_Jornada completa,Tipo_Jornada_Laboral_Tiempo parcial
0,JKLXIQ,51,56857.0,102886,718,338.0,3,6.18,24.0,0.55,...,0.0,0,580.00,1.0,0.0,0.0,0.0,1.0,0.0,0.0
1,4UO7DJ,46,45840.0,30473,689,291.0,1,14.28,36.0,0.34,...,0.0,0,431.25,1.0,0.0,0.0,0.0,1.0,0.0,0.0
2,MO3OJW,36,48156.0,188824,675,161.0,2,13.35,36.0,0.55,...,0.0,0,760.00,0.0,0.0,1.0,0.0,0.0,1.0,0.0
3,I14TTP,42,41472.0,40000,582,312.0,1,11.83,12.0,0.44,...,1.0,0,385.66,0.0,1.0,0.0,0.0,0.0,1.0,0.0
4,8FUIBH,60,70202.0,40743,652,489.0,2,9.27,48.0,0.25,...,1.0,0,535.14,1.0,0.0,0.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
76505,SXTHNM,24,17942.0,40000,823,35.0,2,12.52,36.0,0.55,...,0.0,1,540.62,0.0,0.0,1.0,0.0,1.0,0.0,0.0
76506,0U329X,54,35399.0,14989,709,445.0,1,5.93,48.0,0.47,...,0.0,1,164.30,0.0,0.0,1.0,0.0,0.0,0.0,1.0
76507,MY63P5,28,17390.0,5000,744,92.0,3,16.12,12.0,0.10,...,0.0,0,26.80,0.0,0.0,1.0,0.0,1.0,0.0,0.0
76508,EPB8QX,64,40369.0,40000,680,447.0,2,7.75,12.0,0.19,...,1.0,1,207.42,1.0,0.0,0.0,0.0,0.0,1.0,0.0


In [259]:
filtrado["Estudios"].unique()

<StringArray>
['Grado Universitario', 'Escolar', 'Doctorado', 'Máster']
Length: 4, dtype: str

In [260]:
# 1. Definimos el orden de menor a mayor 
orden_estudios = [["Escolar", "Grado Universitario", "Doctorado", "Máster"]]

# 2. Creamos el modelo y usamos las columnas que queremos darle orden
ordinal_enc = OrdinalEncoder(categories=orden_estudios)

# 3. Ajustamos y transformamos la columna (ademas, se sumamos 1 para que el valor minimo sean 1 y no 0, le convertimos a int)
datos["Estudios"] = (ordinal_enc.fit_transform(datos[["Estudios"]]) + 1).astype(int)

Comprobacion del dataframe despues de los cambios hechos

In [261]:
filtrado

,ID,Edad,Ingresos,Monto_Inicial,Scoring_Crediticio,Meses_Empleo,Num_Creditos,Ratio_Interes,Duracion,Ratio_Deuda_Ingresos,Estudios,Tipo_Jornada_Laboral,Estado_Civil,Posesion_Hipoteca,Personas_Cargo,Fiador,Impago,Prima
0,JKLXIQ,51,56857.0,102886,718,338.0,3,6.18,24.0,0.55,Grado Universitario,Desempleado,Casado,1.0,0.0,0.0,0,580.00
1,4UO7DJ,46,45840.0,30473,689,291.0,1,14.28,36.0,0.34,Escolar,Desempleado,Casado,0.0,0.0,0.0,0,431.25
2,MO3OJW,36,48156.0,188824,675,161.0,2,13.35,36.0,0.55,Grado Universitario,Jornada completa,Soltero,0.0,1.0,0.0,0,760.00
3,I14TTP,42,41472.0,40000,582,312.0,1,11.83,12.0,0.44,Escolar,Jornada completa,Divorciado,1.0,1.0,1.0,0,385.66
4,8FUIBH,60,70202.0,40743,652,489.0,2,9.27,48.0,0.25,Grado Universitario,Tiempo parcial,Casado,1.0,0.0,1.0,0,535.14
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
76510,SXTHNM,24,17942.0,40000,823,35.0,2,12.52,36.0,0.55,Escolar,Desempleado,Soltero,0.0,1.0,0.0,1,540.62
76511,0U329X,54,35399.0,14989,709,445.0,1,5.93,48.0,0.47,Escolar,Tiempo parcial,Soltero,0.0,1.0,0.0,1,164.30
76512,MY63P5,28,17390.0,5000,744,92.0,3,16.12,12.0,0.10,Escolar,Desempleado,Soltero,0.0,0.0,0.0,0,26.80
76513,EPB8QX,64,40369.0,40000,680,447.0,2,7.75,12.0,0.19,Escolar,Jornada completa,Casado,1.0,1.0,1.0,1,207.42


In [262]:
datos

,ID,Edad,Ingresos,Monto_Inicial,Scoring_Crediticio,Meses_Empleo,Num_Creditos,Ratio_Interes,Duracion,Ratio_Deuda_Ingresos,...,Fiador,Impago,Prima,Estado_Civil_Casado,Estado_Civil_Divorciado,Estado_Civil_Soltero,Tipo_Jornada_Laboral_Autónomo,Tipo_Jornada_Laboral_Desempleado,Tipo_Jornada_Laboral_Jornada completa,Tipo_Jornada_Laboral_Tiempo parcial
0,JKLXIQ,51,56857.0,102886,718,338.0,3,6.18,24.0,0.55,...,0.0,0,580.00,1.0,0.0,0.0,0.0,1.0,0.0,0.0
1,4UO7DJ,46,45840.0,30473,689,291.0,1,14.28,36.0,0.34,...,0.0,0,431.25,1.0,0.0,0.0,0.0,1.0,0.0,0.0
2,MO3OJW,36,48156.0,188824,675,161.0,2,13.35,36.0,0.55,...,0.0,0,760.00,0.0,0.0,1.0,0.0,0.0,1.0,0.0
3,I14TTP,42,41472.0,40000,582,312.0,1,11.83,12.0,0.44,...,1.0,0,385.66,0.0,1.0,0.0,0.0,0.0,1.0,0.0
4,8FUIBH,60,70202.0,40743,652,489.0,2,9.27,48.0,0.25,...,1.0,0,535.14,1.0,0.0,0.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
76505,SXTHNM,24,17942.0,40000,823,35.0,2,12.52,36.0,0.55,...,0.0,1,540.62,0.0,0.0,1.0,0.0,1.0,0.0,0.0
76506,0U329X,54,35399.0,14989,709,445.0,1,5.93,48.0,0.47,...,0.0,1,164.30,0.0,0.0,1.0,0.0,0.0,0.0,1.0
76507,MY63P5,28,17390.0,5000,744,92.0,3,16.12,12.0,0.10,...,0.0,0,26.80,0.0,0.0,1.0,0.0,1.0,0.0,0.0
76508,EPB8QX,64,40369.0,40000,680,447.0,2,7.75,12.0,0.19,...,1.0,1,207.42,1.0,0.0,0.0,0.0,0.0,1.0,0.0


In [263]:
datos.info()

<class 'pandas.DataFrame'>
RangeIndex: 76510 entries, 0 to 76509
Data columns (total 23 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   ID                                     76510 non-null  str    
 1   Edad                                   76510 non-null  int64  
 2   Ingresos                               76510 non-null  float64
 3   Monto_Inicial                          76510 non-null  int64  
 4   Scoring_Crediticio                     76510 non-null  int64  
 5   Meses_Empleo                           76510 non-null  float64
 6   Num_Creditos                           76510 non-null  int64  
 7   Ratio_Interes                          76510 non-null  float64
 8   Duracion                               76510 non-null  float64
 9   Ratio_Deuda_Ingresos                   76510 non-null  float64
 10  Estudios                               76510 non-null  int64  
 11  Posesion_Hipo